# MGMT298D: Science and Strategy of AI
## Week 1: Linear Regression & Regularization
### UCLA Anderson School of Management

# <span style="font-size:1.4em;">🗺️ The ML Pipeline at a Glance</span>

<span style="font-size:1.15em;">

This notebook walks through a **complete supervised-learning pipeline** for predicting product sales using regression. Here's the roadmap:

**1. Load & Explore Data** — Read a CSV into a `pandas` DataFrame and inspect it.

**2. Split Data** — Hold out 20% of products so we can measure real out-of-sample performance.

**3. Baseline Model** — Fit plain OLS with minimal features (`sklearn.linear_model.LinearRegression`).

**4. Feature Engineering** — Create lag, rolling, polynomial, and interaction features with `pandas` and `numpy`.

**5. Regularization** — Apply **Lasso**, **Ridge**, and **Elastic Net** (all from `sklearn`) to control overfitting when the feature set grows large.

**6. Model Selection** — Use cross-validation (`LassoCV`, `RidgeCV`, `ElasticNetCV`) to automatically choose the best penalty strength λ.

</span>

# <span style="font-size:1.4em;">📦 Setup & Imports</span>

<span style="font-size:1.15em;">

We begin by importing the key Python packages used throughout:

- **`pandas`** — the standard library for tabular data (`DataFrame`)
- **`numpy`** — numerical computing (arrays, random seeds, math)
- **`matplotlib`** — plotting (used later if needed)
- **`sklearn`** (scikit-learn) — the go-to ML library. We import regression models, scalers, and metrics from it.

</span>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import (LinearRegression, Lasso, Ridge, ElasticNet,
                                  LassoCV, RidgeCV, ElasticNetCV)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

# <span style="font-size:1.4em;">📥 Step 1: Load & Explore the Data</span>

<span style="font-size:1.15em;">

**What's happening:** We use `pandas.read_csv()` to pull the H&M sales dataset directly from a URL into a `DataFrame` — the central data structure in `pandas`. Think of it as a spreadsheet in Python.

**Key call:** `pd.read_csv(url)` — reads a CSV file (local or remote) and returns a DataFrame.

</span>

In [ ]:
url = "https://raw.githubusercontent.com/ucla-anderson-SSAI/SSAI/main/HMData.csv"
df = pd.read_csv(url)
print(f"{len(df)} rows, {df.shape[1]} columns")

# Show all columns in the preview
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
df.head()

## Filter by Product Type

We narrow the dataset to a single product type so that the model learns patterns specific to that category. This is a common first step — start focused, generalize later.

In [ ]:
PRODUCT_TYPE = 'Vest top'  # Change to any product name (see df['name'].unique())

df = df[df['name'] == PRODUCT_TYPE].copy()
print(f"{PRODUCT_TYPE}: {len(df)} rows, {df['id'].nunique()} products")

# <span style="font-size:1.4em;">✂️ Step 2: Train/Test Split</span>

<span style="font-size:1.15em;">

**What's happening:** We split by **product ID** (not by row) so that all monthly observations for a given product stay together. 80% of products go to training, 20% are held out for testing.

**Why it matters:** This prevents data leakage — the model can't peek at future months of the same product it trained on.

**Key calls:** `numpy.random.shuffle()` to randomize the product list, then simple array slicing to partition.

</span>

In [ ]:
np.random.seed(42)
product_ids = df['id'].unique()
np.random.shuffle(product_ids)
split = int(0.8 * len(product_ids))
train_ids, test_ids = product_ids[:split], product_ids[split:]

print(f"Train: {len(train_ids)} products, Test: {len(test_ids)} products")

---
# <span style="font-size:1.4em;">📊 Part 1: Baseline Linear Regression</span>

<span style="font-size:1.15em;">

**What's happening:** We fit the simplest possible regression — **Ordinary Least Squares (OLS)** — using only price and month dummies as features. This establishes a baseline to beat.

**Data flow:**
1. Define the feature list → 2. `StandardScaler().fit_transform()` standardizes features (zero mean, unit variance) → 3. `LinearRegression().fit()` learns the weights → 4. `.predict()` generates predictions → 5. `mean_absolute_error()` measures accuracy.

**Key calls:**
- `StandardScaler()` from `sklearn.preprocessing` — scales features so they're comparable
- `LinearRegression().fit(X, y)` from `sklearn.linear_model` — fits OLS regression
- `mean_absolute_error(y_true, y_pred)` from `sklearn.metrics` — our evaluation metric

</span>

In [ ]:
# Months are already one-hot encoded in the CSV
month_cols = ['January', 'February', 'March', 'April', 'May', 'June',
              'July', 'August', 'September', 'October', 'November', 'December']

basic_features = ['price'] + month_cols
print(f"Features ({len(basic_features)}): {basic_features}")

In [ ]:
def split_and_scale(data, features, train_ids, test_ids):
    """Split by product ID, extract features, and standardize."""
    tr = data[data['id'].isin(train_ids)]
    te = data[data['id'].isin(test_ids)]
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(tr[features])
    X_te = scaler.transform(te[features])
    return X_tr, X_te, tr['sales'].values, te['sales'].values, scaler

X_train, X_test, y_train, y_test, scaler1 = split_and_scale(df, basic_features, train_ids, test_ids)

# Baseline: plain OLS with just price + month dummies
ols_basic = LinearRegression().fit(X_train, y_train)
mae_basic = mean_absolute_error(y_test, ols_basic.predict(X_test))
print(f"OLS (price + month) — Test MAE: {mae_basic:.1f}")

---
# <span style="font-size:1.4em;">🔧 Part 2: Feature Engineering</span>

<span style="font-size:1.15em;">

**What's happening:** We manufacture new input columns from the raw data to give the model more signal. This is where domain knowledge meets `pandas` and `numpy`.

**Features we create:**
- **Lag features** (`groupby().shift()`) — past sales as predictors
- **Rolling statistics** (`rolling().mean()`, `rolling().std()`) — smoothed trends
- **Polynomial** (`price ** 2`) and **interaction** terms (`price * lag_1`) — nonlinear relationships

**Why it matters:** More features can improve predictions, but they also increase the risk of **overfitting**. We'll see the train/test gap widen — motivating regularization in Part 3.

**Key `pandas` calls:** `groupby()`, `shift()`, `rolling()`, `transform()`, `pct_change()`, `fillna()`, `replace()`

</span>

In [ ]:
df_eng = df.copy()

# Lag features: sales from 1, 2, 3 months ago
df_eng['lag_1'] = df_eng.groupby('id')['sales'].shift(1)
df_eng['lag_2'] = df_eng.groupby('id')['sales'].shift(2)
df_eng['lag_3'] = df_eng.groupby('id')['sales'].shift(3)

# Rolling statistics (3-month window, shifted to avoid leakage)
df_eng['ma_3']  = df_eng.groupby('id')['sales'].transform(lambda x: x.rolling(3).mean().shift(1))
df_eng['std_3'] = df_eng.groupby('id')['sales'].transform(lambda x: x.rolling(3).std().shift(1))

# Price features
df_eng['price_pct_change'] = df_eng.groupby('id')['price'].pct_change()
df_eng['price_sq'] = df_eng['price'] ** 2

# Interaction terms
df_eng['price_x_lag_1'] = df_eng['price'] * df_eng['lag_1']
df_eng['lag1_x_lag2']   = df_eng['lag_1'] * df_eng['lag_2']

df_eng.fillna(0, inplace=True)
df_eng.replace([np.inf, -np.inf], 0, inplace=True)

# Color and pattern indicators
color_cols = ['Black', 'Dark Blue', 'White', 'Blue', 'Dark Grey', 'Grey',
              'Light Beige', 'Light Blue', 'Light Pink', 'Beige', 'Dark Red',
              'Greenish Khaki', 'Light Grey', 'Off White', 'Red', 'Pink']
pattern_cols = ['Solid', 'Denim', 'All over pattern', 'Melange', 'Stripe', 'Lace']

# Full feature list
all_features = (['price', 'price_sq', 'price_pct_change',
                 'lag_1', 'lag_2', 'lag_3', 'ma_3', 'std_3',
                 'price_x_lag_1', 'lag1_x_lag2']
                + month_cols + color_cols + pattern_cols)
print(f"{len(all_features)} features: {all_features}")

In [ ]:
X_train2, X_test2, y_train2, y_test2, scaler2 = split_and_scale(df_eng, all_features, train_ids, test_ids)

# OLS with all engineered features — likely overfits
ols_eng = LinearRegression().fit(X_train2, y_train2)
mae_eng_train = mean_absolute_error(y_train2, ols_eng.predict(X_train2))
mae_eng_test  = mean_absolute_error(y_test2, ols_eng.predict(X_test2))

print(f"OLS ({len(all_features)} features)")
print(f"  Train MAE: {mae_eng_train:.1f}")
print(f"  Test  MAE: {mae_eng_test:.1f}")
print(f"  Gap: {mae_eng_test - mae_eng_train:.1f}  (large = overfitting)")

---
# <span style="font-size:1.4em;">⚖️ Part 3: Regularization</span>

<span style="font-size:1.15em;">

**What's happening:** Plain OLS overfits with many features. Regularization adds a **penalty** to the loss function that discourages large weights, forcing the model to be simpler and generalize better.

**The three models (all from `sklearn.linear_model`):**
- **`Lasso`** — penalty shrinks some weights exactly to zero → automatic feature selection
- **`Ridge`** — penalty shrinks all weights toward zero but never eliminates any
- **`ElasticNet`** — a blend of both, controlled by mixing parameter α

**Key parameter:** **λ (lambda)** controls penalty strength. We sweep a grid of λ values to see the tradeoff between underfitting (λ too high) and overfitting (λ too low).

**Key calls:** `Lasso(alpha=λ).fit(X, y)`, `Ridge(alpha=λ).fit(X, y)` — same `.fit()` / `.predict()` API as OLS.

</span>

### Lasso: Shrinks + eliminates weights

Watch the `nonzero` column — as λ increases, Lasso pushes more and more weights to exactly zero. This is **automatic feature selection**.

In [ ]:
# Sweep λ values for Lasso — higher λ = more weights pushed to zero
lambdas = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
lasso_results = []

for lam in lambdas:
    m = Lasso(alpha=lam).fit(X_train2, y_train2)
    lasso_results.append({
        'λ': lam,
        'Train MAE': mean_absolute_error(y_train2, m.predict(X_train2)),
        'Test MAE':  mean_absolute_error(y_test2, m.predict(X_test2)),
        'nonzero':   int(np.sum(m.coef_ != 0))
    })

lasso_df = pd.DataFrame(lasso_results)
print(lasso_df.to_string(index=False))

### Ridge: Shrinks weights but keeps them all

Unlike Lasso, Ridge never zeros out any weight — it shrinks them all proportionally. Compare the Test MAE trajectory to Lasso above.

In [ ]:
# Sweep λ values for Ridge — shrinks weights but never zeros them out
ridge_results = []

for lam in lambdas:
    m = Ridge(alpha=lam).fit(X_train2, y_train2)
    ridge_results.append({
        'λ': lam,
        'Train MAE': mean_absolute_error(y_train2, m.predict(X_train2)),
        'Test MAE':  mean_absolute_error(y_test2, m.predict(X_test2)),
    })

ridge_df = pd.DataFrame(ridge_results)
print(ridge_df.to_string(index=False))

# <span style="font-size:1.4em;">🏆 Step 3: Model Selection via Cross-Validation</span>

<span style="font-size:1.15em;">

**What's happening:** Instead of manually picking λ, we let `sklearn` do it for us. `LassoCV`, `RidgeCV`, and `ElasticNetCV` try many λ values internally using **k-fold cross-validation** on the training set and pick the λ that minimizes error.

**Key calls:**
- `LassoCV(alphas=grid).fit(X, y)` — fits Lasso at every λ in the grid, selects the best via CV
- `RidgeCV(alphas=grid).fit(X, y)` — same for Ridge
- `ElasticNetCV(alphas=grid).fit(X, y)` — same for Elastic Net, also selects α

After fitting, `.alpha_` gives the best λ and `.predict()` uses the best model.

</span>

In [ ]:
# Shared penalty grid for all three CV models
lambdas_cv = np.logspace(-3, 3, 50)

# Use cross-validation to select best λ for Lasso and Ridge, best λ + α for Elastic Net
lasso_cv = LassoCV(alphas=lambdas_cv).fit(X_train2, y_train2)
ridge_cv = RidgeCV(alphas=lambdas_cv).fit(X_train2, y_train2)
enet_cv  = ElasticNetCV(alphas=lambdas_cv).fit(X_train2, y_train2)

# Compare all models
print(f"OLS (price + month)  — Test MAE: {mae_basic:.1f}")
print(f"OLS ({len(all_features)} features)    — Test MAE: {mae_eng_test:.1f}")
print(f"Lasso (λ={lasso_cv.alpha_:.3f})   — Test MAE: {mean_absolute_error(y_test2, lasso_cv.predict(X_test2)):.1f}")
print(f"Ridge (λ={ridge_cv.alpha_:.3f})   — Test MAE: {mean_absolute_error(y_test2, ridge_cv.predict(X_test2)):.1f}")
print(f"ElasticNet (λ={enet_cv.alpha_:.3f}, α={enet_cv.l1_ratio_:.2f}) — Test MAE: {mean_absolute_error(y_test2, enet_cv.predict(X_test2)):.1f}")